In [1]:
from langchain_neo4j import Neo4jGraph
from neo4j import GraphDatabase
import json

In [2]:
medic_graph = Neo4jGraph(
    url="neo4j://127.0.0.1:7687",
    username="neo4j",
    password="Goppanmavane@2"
)

main_driver = GraphDatabase.driver(uri="neo4j://127.0.0.1:7687",auth=("neo4j","Goppanmavane@2"))

In [3]:
def normalize(text):
    if not text:
        return None
    return text.strip().title()

def load_json(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

NUTRIENT ENTITIES EXTRACTION

In [ ]:
def ingest_nutrient(tx, record):

    # print("\n---- NEW RECORD ----")

    nutrient = normalize(record.get("entity"))
    # print("Nutrient:", nutrient)

    if not nutrient:
        print("Skipping due to missing nutrient")
        return

    # Create Nutrient Node
    tx.run("""
    MERGE (n:Nutrients {name:$nutrient})
    """, nutrient=nutrient)

    # ---------------- FOOD SOURCES ----------------
    foods = (record.get("food_sources") or []) + \
        (record.get("natural_sources") or [])

    for food in foods:
        food = normalize(food)
        if not food:
            continue

        tx.run("""
        MERGE (f:FoodnHerb {name:$food})
        SET f.type = 
            CASE 
                WHEN f.type IS NULL THEN "food"
                ELSE f.type
            END
        MATCH (n:Nutrients {name:$nutrient})
        MERGE (n)-[:FOUND_IN]->(f)
        """, food=food, nutrient=nutrient)

    # ---------------- BODY SYSTEMS ----------------
    systems = record.get("target_systems") or \
          record.get("metadata", {}).get("body_systems_supported") or []

    for system in systems:
        system = normalize(system)
        # print(system)

        tx.run("""
        MERGE (b:BodySystem {name:$system})
        MATCH (n:Nutrients {name:$nutrient})
        MERGE (n)-[:SUPPORTS_SYSTEM]->(b)
        """, system=system, nutrient=nutrient)

    # ---------------- INTERACTIONS ----------------
    for other in record.get("interacts_with") or []:
        other = normalize(other)
        # print(other)

        tx.run("""
        MATCH (n1:Nutrients {name:$n1})
        MERGE (n2:Nutrients {name:$n2})
        MERGE (n1)-[:INTERACTS_WITH]-(n2)
        """, n1=nutrient, n2=other)

    # ---------------- DEFICIENCY EFFECTS → Symptom ----------------
    for effect in record.get("deficiency_effects") or []:
        effect = normalize(effect)
        # print(effect)

        tx.run("""
        MERGE (s:Symptom {name:$effect})
        MATCH (n:Nutrients {name:$nutrient})
        MERGE (n)-[:DEFICIENCY_CAUSES]->(s)
        """, effect=effect, nutrient=nutrient)

HERBS ENTITIES EXTRACTION

In [5]:
def ingest_herb(tx, record):

    # ---------------- BASIC INFO ----------------
    name = normalize(record.get("name"))

    if not name:
        print("Skipping herb with no name:", record)
        return

    scientific_name = record.get("scientific_name")
    category = record.get("category")
    description = record.get("description")

    # ---------------- CREATE MAIN NODE ----------------
    tx.run("""
    MERGE (h:FoodnHerb {name:$name})
    SET h.type = 
        CASE 
            WHEN h.type IS NULL THEN "herb"
            WHEN h.type = "food" THEN "herb"
            ELSE h.type
        END,
        h.scientific_name = $scientific_name,
        h.category = $category,
        h.description = $description
    """, name=name, scientific_name=scientific_name,
    category=category, description=description)

    # ---------------- KEY COMPOUNDS ----------------
    for comp in record.get("key_compounds") or []:
        comp = normalize(comp)
        if not comp:
            continue

        tx.run("""
        MATCH (h:FoodnHerb {name:$name})
        MERGE (c:Compound {name:$comp})
        MERGE (h)-[:CONTAINS]->(c)
        """, name=name, comp=comp)

    # ---------------- PROPERTIES ----------------
    properties = set(
        (record.get("primary_properties") or []) +
        (record.get("health_benefits") or [])
    )

    for prop in properties:
        prop = normalize(prop)
        if not prop:
            continue    

        tx.run("""
        MATCH (h:FoodnHerb {name:$name})
        MERGE (p:Property {name:$prop})
        MERGE (h)-[:HAS_PROPERTY]->(p)
        """, name=name, prop=prop)    

    # ---------------- CONDITIONS ----------------
    for cond in record.get("commonly_used_for") or []:
        cond = normalize(cond)
        if not cond:
            continue

        tx.run("""
        MATCH (h:FoodnHerb {name:$name})
        MERGE (c:Symptom {name:$cond})
        MERGE (h)-[:HELPS_WITH]->(c)
        """, name=name, cond=cond)

    # ---------------- COMBINATIONS ----------------
    for combo in record.get("common_combinations") or []:
        combo = normalize(combo)
        if not combo:
            continue

        tx.run("""
        MATCH (h1:FoodnHerb {name:$name})
        MERGE (h2:FoodnHerb {name:$combo})
        SET h2.type = 
            CASE 
                WHEN h2.type IS NULL THEN "food"
                ELSE h2.type
            END
        MERGE (h1)-[:COMBINES_WITH]->(h2)
        """, name=name, combo=combo)

DISEASE ENTITIES EXTRACTION

In [6]:
def ingest_disease(tx, record):

    name = normalize(record.get("name"))

    if not name:
        print("Skipping disease with no name")
        return

    # -------- CREATE DISEASE --------
    tx.run("""
    MERGE (d:Disease {name:$name})
    """, name=name)

    # -------- DISEASE → SYMPTOMS --------
    for symptom in record.get("symptoms") or []:
        symptom = normalize(symptom)
        if not symptom:
            continue

        tx.run("""
        MATCH (d:Disease {name:$name})
        MERGE (s:Symptom {name:$symptom})
        MERGE (d)-[:HAS_SYMPTOM]->(s)
        """, name=name, symptom=symptom)

    # -------- SYMPTOM ← HERBS --------
    for ingredient in record.get("natural_ingredients") or []:
        ingredient = normalize(ingredient)
        if not ingredient:
            continue

        tx.run("""
        MATCH (f:FoodnHerb {name:$ingredient})
        WITH f
        MATCH (d:Disease {name:$name})-[:HAS_SYMPTOM]->(s:Symptom)
        MERGE (f)-[:HELPS_WITH]->(s)
        """, name=name, ingredient=ingredient)

In [7]:
# ---------------- OVERALL PIPELINE FUNCTION ----------------
def run_ingestion(file_path, ingest_function):

    data = load_json(file_path)

    with main_driver.session() as session:
        for record in data:
            session.execute_write(ingest_function, record)

    print(f"✅ Graph created successfully from {file_path}!")

In [8]:
run_ingestion("../datasets/rag_json_docs/nutrients.json", ingest_nutrient)
run_ingestion("../datasets/rag_json_docs/herbs.json", ingest_herb)
run_ingestion("../datasets/rag_json_docs/diseases.json", ingest_disease)

✅ Graph created successfully from ../datasets/rag_json_docs/nutrients.json!
✅ Graph created successfully from ../datasets/rag_json_docs/herbs.json!
✅ Graph created successfully from ../datasets/rag_json_docs/diseases.json!


In [9]:
# data = load_json("../datasets/rag_json_docs/nutrients.json")

# print("Total records:", len(data))  # should be 20

# for i, r in enumerate(data):
#     print(f"\nRecord {i+1}")
#     print("Entity:", r.get("entity"))